<a href="https://colab.research.google.com/github/smile-rr/ocr-fine-app/blob/main/notebooks/02b_stage1_colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02b · Stage 1 · VLM QLoRA（Colab · GPU）

本 notebook 在 Colab 跑 **Qwen2-VL-2B-Instruct** 的 4-bit QLoRA，用 **Unsloth** 加速。

**为什么用 Colab**：Mac 本地用 MLX 跑 VLM LoRA 慢且吃内存；Colab T4 免费额度足够 2B 模型。

**硬件**：Runtime → Change runtime type → T4 GPU（或更强，如 L4 / A100）即可。

**流程**：
1. 检查 GPU
2. 装依赖（Unsloth + trl + peft）
3. HF 登录（Colab Secret 或手输 token）
4. 上传本地准备好的数据 `stage1_colab.zip`
5. 加载模型 + 配 LoRA
6. 训练
7. 推理测试
8. 下载 adapter 回本地（合并用）


## 1. 检查 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('CUDA:', torch.cuda.is_available(), '| bf16:', torch.cuda.is_bf16_supported())

## 2. 安装依赖

> 第一次装大约 3–5 分钟。Unsloth 会自动匹配当前 CUDA 和 PyTorch 版本。

In [ ]:
%%capture
!pip install -U unsloth
!pip install -U datasets trl peft accelerate bitsandbytes pillow

## 3. HF 登录

两种方式（任选）：

**① Colab Secret（推荐，免重复输入）**：左侧 🔑 图标 → 添加 `HF_TOKEN` → 勾「Notebook access」

**② 运行时手输**：运行下面的 cell，用 `getpass` 隐式输入

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print('✅ logged in via Colab secret HF_TOKEN')
except Exception:
    from getpass import getpass
    token = getpass('HF token (hf_...): ')
    login(token=token)
    print('✅ logged in via manual input')

## 4. 上传数据

先在**本地**跑：
```bash
bash scripts/pack_for_colab.sh stage1
```
生成 `stage1_colab.zip`（~80–160 MB，jsonl + 训练图片）。然后运行下面的 cell，在 Colab 里选择文件上传。

In [ ]:
from google.colab import files
up = files.upload()          # 选择 stage1_colab.zip
!unzip -q -o stage1_colab.zip -d /content/
!ls /content/data/stage1_train/ | head
!echo '---' && ls /content/data/stage1_images/ | wc -l && echo 'images'
%cd /content

## 5. 加载 Qwen2-VL-2B（4-bit）+ LoRA

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'Qwen/Qwen2-VL-2B-Instruct',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',   # 省 VRAM
)
FastVisionModel.for_training(model)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,    # 冻住 vision encoder（省 VRAM；图表任务文本侧微调已够）
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0,
    bias='none', random_state=3407,
)
print(model.print_trainable_parameters())

## 6. 转换数据集为 Unsloth vision 格式

本地 sharegpt 格式 → Unsloth 期望的 `messages` 结构（image 是 PIL）。

In [ ]:
import json
from pathlib import Path
from PIL import Image

INSTRUCTION = '请提取图中所有表格，以标准 Markdown 格式输出。如果没有表格，输出 \'无表格\'。'

def convert(row):
    msgs = row['messages']
    img_rel = next(c for c in msgs[0]['content'] if c['type']=='image')['image']
    text    = next(c for c in msgs[0]['content'] if c['type']=='text')['text']
    img = Image.open(Path('/content') / img_rel).convert('RGB')
    return {
        'messages': [
            {'role':'user', 'content':[{'type':'image','image':img},
                                       {'type':'text','text':text}]},
            {'role':'assistant', 'content':[{'type':'text','text':msgs[1]['content']}]},
        ]
    }

raw = [json.loads(l) for l in open('/content/data/stage1_train/train.jsonl')]
train_ds = [convert(r) for r in raw]
print(f'{len(train_ds)} training examples')

## 7. 训练

**参数说明**（全都可调）：
- `per_device_train_batch_size=1` + `gradient_accumulation_steps=4` → 等效 batch 4
- `max_steps=300`：训练步数（~5–15 min on T4）
- `learning_rate=1e-4`：QLoRA 标准值

想更长训练：`max_steps=1000` 或改成 `num_train_epochs=3`。

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=300,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs',
        # VLM 必需：
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        dataset_num_proc=4,
        max_seq_length=2048,
    ),
)
trainer.train()

## 8. 推理验证（微调前 vs 微调后）

In [ ]:
FastVisionModel.for_inference(model)
img = train_ds[0]['messages'][0]['content'][0]['image']
prompt = INSTRUCTION

from transformers import TextStreamer
messages = [{'role':'user','content':[{'type':'image'},{'type':'text','text':prompt}]}]
chat = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(img, chat, return_tensors='pt').to('cuda')
_ = model.generate(**inputs, max_new_tokens=300, streamer=TextStreamer(tokenizer, skip_prompt=True))

## 9. 保存 + 下载 adapter

Adapter 只有 LoRA 权重（几十 MB），下载回本地后用 `scripts/fuse_model.py` 合并到 base 再部署到 Docker。

In [ ]:
model.save_pretrained('stage1_adapter')
tokenizer.save_pretrained('stage1_adapter')
!ls -lh stage1_adapter/
!zip -q -r stage1_adapter.zip stage1_adapter/
!ls -lh stage1_adapter.zip

In [ ]:
from google.colab import files
files.download('stage1_adapter.zip')

---

## 🔁 回到本地部署

1. 把下载的 `stage1_adapter.zip` 解压到项目根目录：
   ```bash
   unzip stage1_adapter.zip -d models/
   mv models/stage1_adapter models/stage1_adapter_hf
   ```
   > 注意：MLX fuse 脚本期望 MLX 格式的 adapter；Colab 产出的是 HF 格式。
   > 若要直接用 transformers 推理（Docker 服务走这条路），改为：
   > - 用 `peft.PeftModel.from_pretrained(base, 'models/stage1_adapter_hf')` 加载
   > - 或者先 `model.merge_and_unload()` 合并后存 HF 格式到 `models/stage1_fused/`

2. 重启 Docker 服务，热加载新 adapter：
   ```bash
   curl -X POST http://localhost:8000/admin/reload \
     -H 'X-Admin-Key: '$ADMIN_API_KEY \
     -d '{"stage":1,"force":true}'
   ```
